In [ ]:
import pandas as pd
# import glob

using glob to read multiple csv files or concat the files when reading

In [2]:
# csv_files = glob.glob("Datasets/*.csv")

# dfs = {file: pd.read_csv(file) for file in csv_files}

# # Access individual DataFrames
# # print(dfs[csv_files[0]].head())
# print(dfs.keys()) # Prints first CSV file's data
# odbc,jdbc



In [21]:
dmatch=pd.read_csv('Datasets/dim_match_summary.csv')
dplayer=pd.read_csv('Datasets/dim_players.csv')
dbat=pd.read_csv('Datasets/fact_bating_summary.csv')
dbow=pd.read_csv('Datasets/fact_bowling_summary.csv')


top 10 batsman in 3 yrs

In [4]:
matchbatsman=dmatch.merge(dbat,left_on='match_id',right_on='match_id')
matchbatsman['matchDate'] = pd.to_datetime(matchbatsman['matchDate'], errors='coerce')
matchbatsman['year']=matchbatsman['matchDate'].dt.year.fillna(0).astype('int64')

# getting only the valid matchdate players
filterbatsman=matchbatsman.query('year !=0')

filterbatsman.groupby('batsmanName')['runs'].sum().reset_index().sort_values(by='runs',ascending=False).head(10)

,batsmanName,runs
66,FafduPlessis,1831
222,ShubmanGill,1812
197,RuturajGaikwad,1567
97,KLRahul,1516
95,JosButtler,1509
216,ShikharDhawan,1392
251,ViratKohli,1385
204,SanjuSamson,1304
230,SuryakumarYadav,1225
69,GlennMaxwell,1214


Top 10 batsmen based on past 3 years batting average. (min 60 balls faced in
each season)

In [5]:
batavg=filterbatsman[filterbatsman['out/not_out']=='out']
top10avg=batavg.groupby(['batsmanName','year']).agg({'runs':'mean','balls':'sum','out/not_out':'count'}).query('balls>=60').reset_index()
top10avg.sort_values(by='runs',ascending=False).round(2).head(10)


,batsmanName,year,runs,balls,out/not_out
50,FafduPlessis,2023,50.08,429,13
170,ShubmanGill,2023,46.64,441,14
69,JosButtler,2022,45.80,472,15
58,HeinrichKlaasen,2023,42.11,220,9
194,YashasviJaiswal,2023,40.54,335,13
145,RuturajGaikwad,2023,40.31,348,13
187,ViratKohli,2023,38.00,347,12
42,DevonConway,2023,38.00,347,12
71,KLRahul,2021,37.70,300,10
34,DavidWarner,2023,36.86,392,14


Top 10 batsmen based on past 3 years strike rate (min 60 balls faced in each
season)

In [6]:
strikerate=filterbatsman.groupby(['batsmanName','year']).agg({'runs':'sum','balls':'sum'}).query('balls>=60').reset_index()
strikerate.assign(strike=lambda x:((x['runs']/x['balls'])*100)).head(10).round(2)

,batsmanName,year,runs,balls,strike
0,ABdeVilliers,2021,313,211,148.34
1,AaronFinch,2022,86,61,140.98
2,AbdulSamad,2021,111,87,127.59
3,AbdulSamad,2023,169,128,132.03
4,AbhinavManohar,2022,108,75,144.00
5,AbhinavManohar,2023,114,83,137.35
6,AbhishekSharma,2021,93,69,134.78
7,AbhishekSharma,2022,426,320,133.12
8,AbhishekSharma,2023,226,157,143.95
9,AidenMarkram,2021,146,119,122.69


Top 10 bowlers based on past 3 years total wickets taken

In [7]:
dbow.groupby('bowlerName')['wickets'].sum().reset_index().sort_values(by='wickets',ascending=False).head(10)

,bowlerName,wickets
110,MohammedShami,67
201,YuzvendraChahal,66
54,HarshalPatel,65
141,RashidKhan,63
20,AveshKhan,47
19,ArshdeepSingh,45
75,KagisoRabada,45
189,VarunChakravarthy,44
163,ShardulThakur,43
181,TrentBoult,42


Top 10 bowlers based on past 3 years bowling average. (min 60 balls bowled in
each season)

In [8]:
bowlers=dmatch.merge(dbow,left_on='match_id',right_on='match_id')
bowlers['matchDate']=pd.to_datetime(bowlers['matchDate'],errors='coerce')
bowlers['year']=bowlers['matchDate'].dt.year.fillna(0).astype('int64')
cleanbowlers=bowlers[bowlers['year'] !=0]
yrtotal=cleanbowlers.groupby(['bowlerName','year']).agg({'overs':'sum','wickets':'sum','runs':'sum'}).reset_index()
yrtotal=yrtotal.assign(bow_avg=lambda x:(x['runs']/x['wickets'])).round(2)
yrtotal.sort_values(by='bow_avg').head(10).query('overs>=10')

,bowlerName,year,overs,wickets,runs,bow_avg
216,MoisesHenriques,2021,10.0,4,45,11.25
189,MarkWood,2023,16.0,11,130,11.82
296,ShahbazAhmed,2021,14.0,7,92,13.14


Top 10 bowlers based on past 3 years economy rate. (min 60 balls bowled in
each season)


In [9]:
totalovr=cleanbowlers.groupby(['bowlerName','year']).agg({'overs':'sum','runs':'sum'}).reset_index()
totalovr['eco_rate']=(totalovr['runs']/totalovr['overs']).round(2)
totalovr.sort_values(by='eco_rate').head(10).query('overs>=10')

,bowlerName,year,overs,runs,eco_rate
216,MoisesHenriques,2021,10.0,45,4.50
318,SunilNarine,2022,56.0,312,5.57
214,MohsinKhan,2022,33.0,197,5.97
97,HarpreetBrar,2021,23.0,139,6.04


Top 5 batsmen based on past 3 years boundary % (fours and sixes).


In [10]:
boundcalc=filterbatsman.groupby('batsmanName').agg({'4s':'sum','6s':'sum','runs':'sum'}).reset_index()
boundaries=boundcalc.assign(totalbound=lambda x:x['4s']+x['6s'],bound=lambda x:(x['totalbound']/x['runs'])*100).round(2)
top5=boundaries.sort_values(by='bound',ascending=False).head(5)
top5

,batsmanName,4s,6s,runs,totalbound,bound
4,AbhijeetTomar,1,0,4,1,25.00
119,LittonDas,1,0,4,1,25.00
220,ShreyasGopal,2,1,16,3,18.75
130,MarkWood,1,1,11,2,18.18
205,SanvirSingh,1,1,11,2,18.18


Top 5 bowlers based on past 3 years dot ball %.

In [11]:
dotbowlers=cleanbowlers.groupby('bowlerName').agg({'overs':'sum','0s':'sum'}).reset_index()
dotbowlers.columns=['bowler','overs','dots']
dotbowlers['totalballs']=dotbowlers['overs']*6
top5dot=dotbowlers.assign(dotpercentage=lambda x:(x['dots']/x['totalballs'])*100).round(2)
top5dot.sort_values(by='dotpercentage',ascending=False).head(5)


,bowler,overs,dots,totalballs,dotpercentage
168,ShreyasIyer,1.0,4,6.0,66.67
58,ImranTahir,4.0,14,24.0,58.33
146,ReeceTopley,2.0,7,12.0,58.33
39,DewaldBrevis,0.3,1,1.8,55.56
171,SimarjeetSingh,18.0,57,108.0,52.78


Top 4 teams based on past 3 years winning %

In [13]:
dmatch['matchDate']=pd.to_datetime(dmatch['matchDate'],errors='coerce')
nondate=dmatch[~dmatch['matchDate'].isnull()]
#calc team1 winners
team1total=nondate[nondate['team1']==nondate['winner']]
team1winner=team1total.groupby('team1')['winner'].count().reset_index()
#calc team2 winners
team2total=nondate[nondate['team2']==nondate['winner']]
team2winner=team2total.groupby('team2')['winner'].count().reset_index()
team2winner.columns=["away","awaywinner"]
#calc total winning for each team
winnerslist=team1winner.merge(team2winner,left_on='team1',right_on='away')
winners=winnerslist.assign(totalwin=lambda x:x['winner']+x['awaywinner'])
#  total match by teamwise
team1match=dmatch.groupby('team1').size().reset_index(name='team1count')
team2match=dmatch.groupby('team2').size().reset_index(name='team2count')
totalmatches=team1match.merge(team2match,left_on='team1',right_on='team2')
totalmatches['teamtotal']=totalmatches['team1count']+totalmatches['team2count']
#calc winning percentage for each team
total=winners.merge(totalmatches,left_on='team1',right_on='team2')
total['winningpercent']=((total['totalwin']/total['teamtotal'])*100).round(2)
top4=total[['team2','totalwin','teamtotal','winningpercent']]
top4.sort_values(by='winningpercent',ascending=False).head(4)



,team2,totalwin,teamtotal,winningpercent
9,Titans,23,33,69.70
7,Super Giants,17,29,58.62
4,RCB,25,45,55.56
8,Super Kings,24,45,53.33


Top 2 teams with the highest number of wins achieved by chasing targets over
the past 3 years.

In [20]:
chasingteams=nondate[['team2','winner']]
chasewinners=chasingteams.query('team2==winner')
chasewinners.groupby('team2').size().reset_index(name='count').sort_values(by='count',ascending=False).head(2)

,team2,count
0,Capitals,14
1,KKR,14
